# Clase 085 — Random Forests y Extra Trees

Ensambles de árboles **decorrelacionados**: Random Forest (subsampling de features +
thresholds óptimos) vs Extra Trees (thresholds **aleatorios**). Comparamos accuracy,
tiempo, OOB y sensibilidad a `max_features`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset `load_breast_cancer`

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape)

## 2. Baseline: árbol único

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, tree.predict(X_test))
print(f'Árbol único acc test: {acc_tree:.4f}')

## 3. Random Forest vs Extra Trees (accuracy y tiempo)

In [ ]:
def fit_time(model):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    return time.perf_counter() - t0

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
et = ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=1)

t_rf = fit_time(rf)
t_et = fit_time(et)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
acc_et = accuracy_score(y_test, et.predict(X_test))

print(f'RandomForest  acc {acc_rf:.4f}  tiempo {t_rf:.3f}s')
print(f'ExtraTrees    acc {acc_et:.4f}  tiempo {t_et:.3f}s')
assert acc_rf > acc_tree, 'el bosque debería superar al árbol único'
print('assert OK: el ensemble supera al árbol único (decorrelación reduce varianza)')

## 4. OOB score como reemplazo de CV

In [ ]:
rf_oob = RandomForestClassifier(
    n_estimators=200, oob_score=True, bootstrap=True,
    random_state=42, n_jobs=1)
rf_oob.fit(X_train, y_train)
acc_rf_test = accuracy_score(y_test, rf_oob.predict(X_test))
print(f'oob_score_    : {rf_oob.oob_score_:.4f}')
print(f'accuracy test : {acc_rf_test:.4f}')
assert abs(rf_oob.oob_score_ - acc_rf_test) < 0.06, 'OOB debería aproximar el test'
print('assert OK: OOB aproxima el error de generalización, gratis')

## 5. Sensibilidad a `max_features` (CV)

In [ ]:
opciones = [1, 'sqrt', 'log2', 0.5, 1.0]
medias = []
for mf in opciones:
    rf_mf = RandomForestClassifier(
        n_estimators=200, max_features=mf, random_state=42, n_jobs=1)
    scores = cross_val_score(rf_mf, X_train, y_train, cv=5, n_jobs=1)
    medias.append(scores.mean())
    print(f'max_features={str(mf):6s} -> CV acc {scores.mean():.4f}')

mejor = opciones[int(np.argmax(medias))]
print(f'\nmejor max_features: {mejor}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(o) for o in opciones], medias, color='#37a')
ax.set_ylim(min(medias) - 0.01, max(medias) + 0.01)
ax.set_ylabel('CV accuracy (5-fold)')
ax.set_xlabel('max_features')
ax.set_title('Bajar max_features decorrelaciona los árboles')
plt.tight_layout()
plt.show()

## Ejercicios

1. Pedí `oob_score=True` con `bootstrap=False` y observá el error: OOB requiere
   muestras fuera del bootstrap.
2. Repetí el análisis con `RandomForestRegressor`/`ExtraTreesRegressor` sobre
   `make_regression` y reportá `R²`.
3. Variá `n_estimators` en {50, 100, 300, 500} y graficá la curva de accuracy vs tiempo.
4. Comparativa de tiempos: en promedio, ¿ExtraTrees entrena más rápido que RF? ¿Por qué?

## Conclusiones

- Random Forest = bagging de árboles + subsampling de features **en cada split**.
- Extra Trees añade thresholds aleatorios: más varianza reducida, algo más de sesgo,
  y suele ser más rápido.
- `max_features` bajo decorrelaciona y suele mejorar; `sqrt` es un default sólido en
  clasificación.
- El OOB estima generalización sin CV, gratis (solo con `bootstrap=True`).